<a href="https://colab.research.google.com/github/KirilIKochetkov/-/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics opencv-python-headless pillow scikit-learn
import cv2
from ultralytics import YOLO
import numpy as np
from collections import Counter
from sklearn.cluster import KMeans
from google.colab.patches import cv2_imshow

model = YOLO('yolov8n.pt')

image_path = '1-1.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def get_dominant_color(roi, k=3):
    if roi.size == 0:
        return "unknown"

    pixels = roi.reshape(-1, 3)

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels)

    colors = kmeans.cluster_centers_
    labels = kmeans.labels_

    label_counts = Counter(labels)
    dominant_cluster = max(label_counts, key=label_counts.get)
    dominant_color = colors[dominant_cluster]

    return dominant_color

def rgb_to_color_name(rgb_color):

    r, g, b = rgb_color

    if r > 200 and g > 200 and b > 200:
        return "white"
    elif r < 60 and g < 60 and b < 60:
        return "black"
    elif r > 200 and g < 100 and b < 100:
        return "red"
    elif r < 100 and g > 150 and b < 100:
        return "green"
    elif r < 100 and g < 150 and b > 180:
        return "blue"
    elif r > 200 and g > 180 and b < 100:
        return "yellow"
    elif r > 180 and g > 120 and b < 100:
        return "orange"
    elif r > 150 and g < 100 and b > 180:
        return "purple"
    elif abs(r-g) < 30 and abs(g-b) < 30 and abs(r-b) < 30:
        if r > 150:
            return "silver"
        else:
            return "gray"
    elif r > 150 and g > 150 and b > 150:
        return "light"
    else:
        if r > g and r > b:
            if r - g > 50:
                return "red"
            else:
                return "orange/red"
        elif g > r and g > b:
            return "green"
        elif b > r and b > g:
            if b - r > 50:
                return "blue"
            else:
                return "blue/gray"
        else:
            return "other"


def preprocess_car_roi(roi):

    lab = cv2.cvtColor(roi, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)

    # Удаляем шум
    denoised = cv2.fastNlMeansDenoisingColored(enhanced, None, 10, 10, 7, 21)

    return denoised

def extract_main_body(roi):
    h, w, _ = roi.shape

    center_h_start = int(h * 0.3)
    center_h_end = int(h * 0.7)
    center_w_start = int(w * 0.2)
    center_w_end = int(w * 0.8)

    if center_h_end > center_h_start and center_w_end > center_w_start:
        body_roi = roi[center_h_start:center_h_end, center_w_start:center_w_end]
        if body_roi.size > 0:
            return body_roi

    return roi


results = model(image_rgb, conf=0.3)

cars = []
car_colors = []
car_color_values = []

for result in results:
    for box in result.boxes:

        if int(box.cls) == 2:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            car_roi = image_rgb[y1:y2, x1:x2]

            if car_roi.size == 0:
                continue

            processed_roi = preprocess_car_roi(car_roi)

            body_roi = extract_main_body(processed_roi)

            dominant_rgb = get_dominant_color(body_roi)
            color_name = rgb_to_color_name(dominant_rgb)

            car_colors.append(color_name)
            car_color_values.append(dominant_rgb)

            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

            label = f'Car: {color_name}'
            cv2.putText(image, label, (x1, y1-25),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)


color_count = Counter(car_colors)
total_cars = len(car_colors)

print("="*50)
print(f"НАЙДЕНО АВТОМОБИЛЕЙ: {total_cars}")
print("="*50)
print("РАСПРЕДЕЛЕНИЕ ПО ЦВЕТАМ:")
print("-"*30)

for color, count in sorted(color_count.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_cars) * 100
    print(f"  {color:12} : {count:2} ({percentage:.1f}%)")

print("="*50)

if total_cars > 0:
    print("\nИНФОРМАЦИЯ:")
    for i, (color, rgb) in enumerate(zip(car_colors, car_color_values), 1):
        print(f"  Авто {i}: {color}")

# Отображение результата
cv2_imshow(image)


